# Global Step 1: Dataset Ingestion from Google Drive (`gdown`) & AWS S3 Sync

**RUN ONCE:** This notebook downloads the dataset directly from Google Drive using `gdown`, extracts it locally to `dataset/`, deletes the zip file to free up disk space on the 16 GB EBS limit, and syncs the dataset to your Amazon S3 bucket.

Once executed, all models and training notebooks can re-use the dataset from disk or S3 without downloading it again.

In [ ]:
# 1. Install required packages directly into active SageMaker kernel
%pip install -q gdown boto3 pandas tqdm unidecode indic-transliteration

In [ ]:
import os
import sys
import zipfile
import gdown
import boto3

# Base directory configuration
base_dir = os.path.dirname(os.path.dirname(os.path.abspath("")))
dataset_dir = os.path.join(base_dir, "dataset")
s3_bucket = "amazon-ml-challenge-2026-entity-resolution"
s3_prefix = "dataset"

print(f"Base Directory   : {base_dir}")
print(f"Dataset Directory: {dataset_dir}")
print(f"S3 Target Bucket : s3://{s3_bucket}/{s3_prefix}/")

## 2. Download Dataset from Google Drive
Paste your Google Drive shareable file link or File ID below.

In [ ]:
# PASTE YOUR GOOGLE DRIVE ID OR SHAREABLE LINK HERE
GDRIVE_URL = "https://drive.google.com/drive/folders/1G8EJe2gARnLJXj12GzWf5lR-QD364Fnp?usp=sharing"
GDRIVE_ID = "1G8EJe2gARnLJXj12GzWf5lR-QD364Fnp"

download_zip_path = os.path.join(base_dir, "dataset.zip")

try:
    from approach_1_contrastiveVAE.src.s3_utils import sync_directory_from_s3
except ImportError:
    from src.s3_utils import sync_directory_from_s3

# STEP A: Check if dataset exists locally or can be fetched directly from Amazon S3
print("Checking if dataset exists locally or can be fetched from Amazon S3...")
s3_success = False

if os.path.exists(dataset_dir) and os.path.exists(os.path.join(dataset_dir, "train")):
    s3_success = True
    print(f"Dataset already exists locally at {dataset_dir} — ready to proceed!")
else:
    print(f"Dataset not found locally. Fetching directly from S3 bucket s3://{s3_bucket}/{s3_prefix}...")
    s3_success = sync_directory_from_s3(bucket=s3_bucket, s3_prefix=s3_prefix, local_dir=dataset_dir)

# STEP B: If S3 does not have the files yet, fall back to Google Drive (gdown)
if not s3_success:
    print("\nDataset not found in S3. Falling back to Google Drive download via gdown...")
    success = False
    
    # Attempt 1: Try gdown folder download using full URL
    try:
        print(f"Attempt 1: Downloading Google Drive Folder ({GDRIVE_URL})...")
        gdown.download_folder(url=GDRIVE_URL, output=dataset_dir, quiet=False)
        if os.path.exists(os.path.join(dataset_dir, "train")):
            success = True
            print("Folder download succeeded!")
    except Exception as e1:
        print(f"Attempt 1 failed: {e1}")

    # Attempt 2: Try folder download using folder ID
    if not success:
        try:
            print("Attempt 2: Downloading Google Drive Folder using ID...")
            gdown.download_folder(id=GDRIVE_ID, output=dataset_dir, quiet=False)
            if os.path.exists(os.path.join(dataset_dir, "train")):
                success = True
                print("Folder download succeeded!")
        except Exception as e2:
            print(f"Attempt 2 failed: {e2}")

    # Attempt 3: Try file download with fuzzy matching
    if not success:
        try:
            print("Attempt 3: Downloading as Zip File...")
            file_url = f"https://drive.google.com/uc?id={GDRIVE_ID}"
            gdown.download(file_url, download_zip_path, quiet=False, fuzzy=True)
            if os.path.exists(download_zip_path) and zipfile.is_zipfile(download_zip_path):
                print(f"Extracting {download_zip_path}...")
                with zipfile.ZipFile(download_zip_path, 'r') as zip_ref:
                    zip_ref.extractall(base_dir)
                os.remove(download_zip_path)
                success = True
                print("Zip file extracted successfully!")
        except Exception as e3:
            print(f"Attempt 3 failed: {e3}")

    if not success:
        print("\n" + "="*70)
        print("ACTION REQUIRED: DATASET NOT FOUND IN S3 OR GOOGLE DRIVE")
        print("="*70)
        print("Please upload the 'dataset/' folder (containing 'train/' and 'test/' TSVs)")
        print(f"directly into SageMaker at location: {dataset_dir}")
        print("or upload directly to your S3 bucket: s3://" + s3_bucket + "/" + s3_prefix + "/")
        print("="*70)

## 3. Sync Dataset to Amazon S3 Bucket
Uploads raw dataset files (`train_source1/2/3.tsv` and `test_source1/2/3.tsv`) to Amazon S3.

In [ ]:
def sync_directory_to_s3(local_dir: str, bucket: str, prefix: str) -> None:
    s3_client = boto3.client("s3")
    for root, _, files in os.walk(local_dir):
        for file in files:
            local_path = os.path.join(root, file)
            relative_path = os.path.relpath(local_path, local_dir)
            s3_key = os.path.join(prefix, relative_path).replace("\\", "/")
            try:
                s3_client.upload_file(local_path, bucket, s3_key)
                print(f"Uploaded {relative_path} -> s3://{bucket}/{s3_key}")
            except Exception as e:
                print(f"Failed to upload {local_path}: {e}")
    print(f"\n--> Successfully synced {local_dir} to s3://{bucket}/{prefix}/")

if os.path.exists(dataset_dir):
    sync_directory_to_s3(local_dir=dataset_dir, bucket=s3_bucket, prefix=s3_prefix)
else:
    print(f"Dataset directory not found at {dataset_dir}")